# Monte Carlo Simulation for European Option Pricing

This notebook derives and implements risk-neutral Monte Carlo pricing under geometric Brownian motion, then compares the estimates with Black--Scholes.

## 1. Model

Under the risk-neutral measure, $dS_t=rS_tdt+\sigma S_tdW_t$, hence
$$S_T=S_0\exp\left((r-\tfrac12\sigma^2)T+\sigma\sqrt{T}Z\right),\quad Z\sim N(0,1).$$
For a payoff $H(S_T)$, $V_0=e^{-rT}\mathbb{E}^{\mathbb{Q}}[H(S_T)]$.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from option_pricing import (OptionParameters, black_scholes_price,
                            monte_carlo_price, simulate_gbm_paths)

params = OptionParameters(spot=100, strike=100, maturity=1,
                          rate=0.05, volatility=0.20)

## 2. Sample risk-neutral paths

In [ ]:
paths = simulate_gbm_paths(params, n_paths=30, n_steps=252, seed=7)
times = np.linspace(0, params.maturity, paths.shape[1])
plt.figure(figsize=(10, 5))
plt.plot(times, paths.T, alpha=0.7, linewidth=0.9)
plt.axhline(params.strike, color='black', linestyle='--', label='Strike')
plt.xlabel('Time (years)'); plt.ylabel('Stock price')
plt.title('Risk-neutral geometric Brownian motion paths'); plt.legend();


## 3. Monte Carlo estimates and analytical benchmarks

In [ ]:
for option_type in ('call', 'put'):
    mc = monte_carlo_price(params, option_type, n_simulations=500_000, seed=2026)
    bs = black_scholes_price(params, option_type)
    print(f'{option_type.capitalize()}: MC={mc.price:.4f}, BS={bs:.4f}, '
          f'SE={mc.standard_error:.4f}, 95% CI={mc.confidence_interval}')

## 4. Convergence experiment

The central limit theorem predicts standard error proportional to $N^{-1/2}$.

In [ ]:
counts = np.array([1_000, 5_000, 10_000, 50_000, 100_000, 500_000])
bs_call = black_scholes_price(params, 'call')
estimates, lower, upper = [], [], []
for i, n in enumerate(counts):
    result = monte_carlo_price(params, 'call', n_simulations=int(n), seed=42+i)
    estimates.append(result.price)
    lower.append(result.confidence_interval[0])
    upper.append(result.confidence_interval[1])

plt.figure(figsize=(8, 5))
plt.plot(counts, estimates, 'o-', label='Monte Carlo')
plt.fill_between(counts, lower, upper, alpha=0.2, label='95% CI')
plt.axhline(bs_call, color='black', linestyle='--', label='Black–Scholes')
plt.xscale('log'); plt.xlabel('Number of simulations')
plt.ylabel('Call price'); plt.title('Monte Carlo convergence'); plt.legend();

## 5. Variance reduction with antithetic variates

Pairing each normal shock $Z$ with $-Z$ preserves unbiasedness and often lowers variance.

In [ ]:
plain = monte_carlo_price(params, 'call', n_simulations=100_000, seed=10)
anti = monte_carlo_price(params, 'call', n_simulations=100_000, seed=10,
                         antithetic=True)
print(f'Plain SE: {plain.standard_error:.5f}')
print(f'Antithetic SE: {anti.standard_error:.5f}')
print(f'SE reduction: {1 - anti.standard_error/plain.standard_error:.1%}')

## Conclusion

The experiment links the GBM SDE to risk-neutral valuation. Monte Carlo is flexible and statistically interpretable, while Black--Scholes supplies an exact benchmark in this special setting. The main computational cost is the slow $N^{-1/2}$ convergence rate.